In [6]:
import sys
print(sys.version)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


### CÀI ĐẶT HÀM

In [7]:
import numpy as np

def dinh_thuc(M, tol=1e-12): 
   
    M = np.array(M, dtype=float)  # tạo bản sao để không làm thay đổi ma trận gốc

    if M.ndim != 2 or M.shape[0] != M.shape[1]: # kiểm tra: ma trận là ma trận vuông nếu M là mảng 2 chiều và M có số hàng bằng số cột 
        raise ValueError("Định thức chỉ tính cho ma trận vuông.") 

    # khởi tạo
    n = M.shape[0]
    det = 1.0
    #chuyển ma trận M thành ma trận tam giác trên
    #-> det(M) = tích của các phần tử thuộc đường chéo chính
    
    for i in range(n):
        # Chọn phần tử chính (phần tử có giá trị max) theo cột
        p = i + np.argmax(np.abs(M[i:, i]))
        
        # nếu phần tử trụ quá nhỏ -> trụ xấp xỉ 0 -> suy biến
        if abs(M[p, i]) < tol:
            return 0.0

        # Đổi dòng thì định thức đổi dấu
        if p != i:
            M[[i, p]] = M[[p, i]]
            det = -det

        # tích các phần tử trụ (phần tử thuộc đường chéo chính)
        det *= M[i, i]

        # Khử các phần tử phía dưới phần tử chính (khử gauss)
        for j in range(i + 1, n):
            he_so = M[j, i] / M[i, i]
            M[j, i:] -= he_so * M[i, i:]

    return det


def conjugate_gradient(A, b, x0, eps) -> tuple[np.ndarray, int]:
    """
    A: Ma trận vuông, đối xứng xác định dương.
    b: Vectơ vế phải.
    x0: Vectơ nghiệm ban đầu.
    eps: Sai số cho phép, eps > 0.
    """

    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    x = np.asarray(x0, dtype=float).copy()

    # Kiểm tra ma trận vuông
    if A.ndim != 2 or A.shape[0] != A.shape[1] or A.shape[0] == 0:
        raise ValueError("A phải là ma trận vuông không rỗng.")

    # lưu cấp của A vào n
    n = A.shape[0]

    # Kiểm tra ma trận đối xứng
    if not np.allclose(A, A.T): # so sánh bằng giữ A vs A^T
        raise ValueError("A phải là ma trận đối xứng.")

    # Kiểm tra xác định dương 
    for k in range(1, n + 1):
        det_k = dinh_thuc(A[:k, :k]) # tính định thức con lấy ra từ k hàng và cột của A
        if det_k <= 0:
            raise ValueError("A không phải ma trận xác định dương ")

    # Khởi tạo phần dư (r) và hướng tìm kiếm (s)
    r = b - np.dot(A, x)
    s = r.copy()

    print("Nghiệm ban đầu x0:", x0)
    print("Phần dư ban đầu r0:", r)

    # kiểm tra nghiệm ban đầu 
    if np.sqrt(np.dot(r, r)) < eps:
        print("Nghiệm ban đầu đã thỏa mãn điều kiện dừng.")
        return x, 0

    # vòng lặp
    for k in range(1, n + 1):

        # Tính alpha = s.r/s.As
        As = np.dot(A, s)
        mau = np.dot(s, As)
        if mau <= 0: # vì ma trận xác định dương nên mẫu phải dương
            raise ValueError("Mẫu số không hợp lệ.")

        alpha = np.dot(s, r) / mau

        # Cập nhật nghiệm, phần dư và chuẩn phần dư để xét điều kiện dừng
        x = x + alpha * s
        r_moi = b - np.dot(A, x)
        chuan_r = np.sqrt(np.sum(r_moi**2))

        # Hiển thị kết quả từng lần lặp
        print(f"\nLần lặp thứ {k}:")
        print("Nghiệm x =", [round(float(i), 6) for i in x])
        print("Phần dư r =", [round(float(i), 6) for i in r_moi])
        print("Chuẩn phần dư ||r|| =", round(float(chuan_r), 6))

        # Kiểm tra điều kiện dừng
        if chuan_r < eps:
            print(f"Thỏa mãn sau {k} lần lặp.")
            return x, k
            
        # Tính beta
        beta = -np.dot(r_moi, As) / mau

        # Cập nhật hướng tìm kiếm
        s = r_moi + beta * s
        r = r_moi.copy()
        
    return x, n

In [8]:
# Ví dụ 4.10
A = np.array([[4, -1,  1],
              [-1, 4, -2],
              [1, -2,  4]], dtype=float)
b = np.array([12, -1, 5], dtype=float)
kq = conjugate_gradient(A, b, x0 = np.array([0, 0, 0]) , eps = 1e-4)

Nghiệm ban đầu x0: [0 0 0]
Phần dư ban đầu r0: [12. -1.  5.]

Lần lặp thứ 1:
Nghiệm x = [2.417062, -0.201422, 1.007109]
Phần dư r = [1.123223, 4.236967, -1.848341]
Chuẩn phần dư ||r|| = 4.757088

Lần lặp thứ 2:
Nghiệm x = [3.077523, 0.794832, 0.719982]
Phần dư r = [-0.235242, 0.33816, 0.632212]
Chuẩn phần dư ||r|| = 0.754575

Lần lặp thứ 3:
Nghiệm x = [3.0, 1.0, 1.0]
Phần dư r = [-0.0, 0.0, -0.0]
Chuẩn phần dư ||r|| = 0.0
Thỏa mãn sau 3 lần lặp.


In [9]:
# Xét các ví dụ
# Ma trận cấp 3

A3 = np.array([
    [4, -1,  1],
    [-1, 4, -2],
    [1, -2,  4]
], dtype=float)

b3 = np.array([12, -1, 5], dtype=float)
x03 = np.zeros(3)

print("\n=== MA TRẬN CẤP 3")
conjugate_gradient(A3, b3, x03, eps = 1e-8)

# Ma trận cấp 4

A4 = np.array([
    [4, -1,  0,  0],
    [-1, 4, -1,  0],
    [0, -1,  4, -1],
    [0,  0, -1,  4]
], dtype=float)

b4 = np.array([2, 4, 6, 13], dtype=float)
x04 = np.zeros(4)

print("\n==== MA TRẬN CẤP 4 ====")
conjugate_gradient(A4, b4, x04, eps = 1e-8)

# Ma trận cấp 5

A5 = np.array([
    [4, -1,  0,  0,  0],
    [-1, 4, -1,  0,  0],
    [0, -1,  4, -1,  0],
    [0,  0, -1,  4, -1],
    [0,  0,  0, -1,  4]
], dtype=float)

b5 = np.array([2, 4, 6, 8, 16], dtype=float)
x05 = np.zeros(5)

print("\n=== MA TRẬN CẤP 5 ====")
conjugate_gradient(A5, b5, x05, eps = 1e-8)


=== MA TRẬN CẤP 3
Nghiệm ban đầu x0: [0. 0. 0.]
Phần dư ban đầu r0: [12. -1.  5.]

Lần lặp thứ 1:
Nghiệm x = [2.417062, -0.201422, 1.007109]
Phần dư r = [1.123223, 4.236967, -1.848341]
Chuẩn phần dư ||r|| = 4.757088

Lần lặp thứ 2:
Nghiệm x = [3.077523, 0.794832, 0.719982]
Phần dư r = [-0.235242, 0.33816, 0.632212]
Chuẩn phần dư ||r|| = 0.754575

Lần lặp thứ 3:
Nghiệm x = [3.0, 1.0, 1.0]
Phần dư r = [-0.0, 0.0, -0.0]
Chuẩn phần dư ||r|| = 0.0
Thỏa mãn sau 3 lần lặp.

==== MA TRẬN CẤP 4 ====
Nghiệm ban đầu x0: [0. 0. 0. 0.]
Phần dư ban đầu r0: [ 2.  4.  6. 13.]

Lần lặp thứ 1:
Nghiệm x = [0.661765, 1.323529, 1.985294, 4.301471]
Phần dư r = [0.676471, 1.352941, 3.683824, -2.220588]
Chuẩn phần dư ||r|| = 4.559565

Lần lặp thứ 2:
Nghiệm x = [0.88035, 1.7607, 3.060931, 4.04275]
Phần dư r = [0.2393, 0.898482, -0.440273, -0.110068]
Chuẩn phần dư ||r|| = 1.034645

Lần lặp thứ 3:
Nghiệm x = [0.950491, 2.004813, 3.006023, 4.002551]
Phần dư r = [0.202847, -0.062736, -0.01673, -0.004182]
Chuẩn ph

(array([1., 2., 3., 4., 5.]), 5)

In [10]:
import pytest
import numpy as np
import io
import contextlib

# Chạy file này SAU ô chứa chương trình (đã định nghĩa dinh_thuc và conjugate_gradient)

# assert.... thỏa mãn nếu điều kiện đúng, sai thì test thất bại
# Phương pháp CG: r0 = b - A x0, s0 = r0
#   alpha = (s.r)/(s.As), x_moi = x + alpha*s, r_moi = b - A x_moi
#   beta = -(r_moi.As)/(s.As), s_moi = r_moi + beta*s
# CG hội tụ (về lý thuyết) sau tối đa n bước với A đối xứng xác định dương cấp n


# hàm tiện ích 
def bat_output(A, b, x0, eps):
    """Chạy conjugate_gradient, bắt phần in ra màn hình. Trả về (x, k, output)."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        x, k = conjugate_gradient(A, b, x0, eps)
    return x, k, buf.getvalue()


def lay_dong(out):
    """Tách output thành các dòng (bỏ dòng trống, bỏ khoảng trắng thừa)."""
    return [d.strip() for d in out.splitlines() if d.strip()]


# PHẦN 1: dinh_thuc

# 1 ví dụ chuẩn (tính tay được)
def test_dinh_thuc_2x2():
    # A = 1  2
    #     3  4
    # det = 1*4 - 2*3 = -2
    assert dinh_thuc([[1, 2], [3, 4]]) == pytest.approx(-2.0)


# 2 ma trận 1x1
def test_dinh_thuc_1x1():
    assert dinh_thuc([[5]]) == pytest.approx(5.0)


# 3 ma trận đường chéo, det = tích đường chéo
def test_dinh_thuc_duong_cheo():
    # diag(2,3,4) -> det = 2*3*4 = 24
    assert dinh_thuc(np.diag([2, 3, 4])) == pytest.approx(24.0)


# 4 ma trận đơn vị
def test_dinh_thuc_don_vi():
    assert dinh_thuc(np.eye(4)) == pytest.approx(1.0)


# 5 phải đổi dòng -> định thức đổi dấu
def test_dinh_thuc_doi_dong():
    # A = 0  1
    #     1  0
    # M[0,0] = 0 nên phải đổi dòng 0 với dòng 1, det = -(1*1) = -1
    assert dinh_thuc([[0, 1], [1, 0]]) == pytest.approx(-1.0)


# 6 ma trận suy biến -> trả về 0
def test_dinh_thuc_suy_bien():
    # hai dòng tỉ lệ: dòng 2 = 2 * dòng 1
    assert dinh_thuc([[1, 2], [2, 4]]) == 0.0
    # có dòng toàn 0
    assert dinh_thuc([[1, 2, 3], [0, 0, 0], [4, 5, 6]]) == 0.0


# 7 so sánh với numpy trên nhiều ma trận ngẫu nhiên
def test_dinh_thuc_so_voi_numpy():
    rng = np.random.default_rng(0)
    for n in [2, 3, 4, 5, 8]:
        for _ in range(50):
            M = rng.standard_normal((n, n))
            assert dinh_thuc(M) == pytest.approx(np.linalg.det(M), rel=1e-6, abs=1e-8)


# 8 không làm thay đổi ma trận gốc (hàm đã copy bằng np.array)
def test_dinh_thuc_khong_doi_ma_tran_goc():
    M = np.array([[0.0, 1.0], [2.0, 3.0]])
    goc = M.copy()
    dinh_thuc(M)
    assert np.array_equal(M, goc)


# 9 tham số tol: pivot nhỏ hơn tol thì coi là suy biến
def test_dinh_thuc_tol():
    # pivot lớn nhất = 1e-3
    assert dinh_thuc([[1e-3, 0], [0, 1e-3]], tol=1e-2) == 0.0   # 1e-3 < 1e-2 -> 0
    assert dinh_thuc([[1e-3, 0], [0, 1e-3]], tol=1e-6) == pytest.approx(1e-6)


# 10 phải báo lỗi với ma trận không vuông / không phải 2 chiều
def test_dinh_thuc_bao_loi():
    with pytest.raises(ValueError):
        dinh_thuc([[1, 2, 3], [4, 5, 6]])   # 2x3
    with pytest.raises(ValueError):
        dinh_thuc([1, 2, 3])                # 1 chiều

        
# PHẦN 2: conjugate_gradient

# 11 ví dụ chuẩn (tính tay được)
def test_cg_vi_du_chuan():
    # A = 4  1     b = 1
    #     1  3         2
    # det con cấp 1 = 4 > 0, cấp 2 = 12 - 1 = 11 > 0 -> A xác định dương
    # nghiệm: 4x + y = 1 ; x + 3y = 2 -> x = 1/11, y = 7/11
    x, k, _ = bat_output([[4, 1], [1, 3]], [1, 2], [0, 0], 1e-8)
    assert x == pytest.approx([1 / 11, 7 / 11], abs=1e-8)
    assert k <= 2   # tối đa n = 2 bước


# 12 kiểm tra tay đúng 1 bước lặp
def test_cg_mot_buoc_lap_tinh_tay():
    # x0 = 0 -> r0 = b = (1,2), s0 = (1,2)
    # As0 = (4*1+1*2, 1*1+3*2) = (6,7)
    # s.As = 1*6 + 2*7 = 20 ; s.r = 1*1 + 2*2 = 5 -> alpha = 5/20 = 0.25
    # x1 = 0.25*(1,2) = (0.25, 0.5)
    # r1 = b - A x1 = (1-(1+0.5), 2-(0.25+1.5)) = (-0.5, 0.25)
    # ||r1|| = căn(0.25+0.0625) = 0.559017
    # chọn eps = 0.6 > 0.559 nên dừng ngay sau lần lặp thứ 1 (còn ||r0|| = căn5 = 2.236 > 0.6)
    x, k, _ = bat_output([[4, 1], [1, 3]], [1, 2], [0, 0], 0.6)
    assert k == 1
    assert x == pytest.approx([0.25, 0.5])


# 13 ma trận đường chéo
def test_cg_duong_cheo():
    # A = diag(1,2,3), b = (1,4,9) -> x = (1,2,3)
    x, k, _ = bat_output(np.diag([1.0, 2.0, 3.0]), [1, 4, 9], [0, 0, 0], 1e-10)
    assert x == pytest.approx([1, 2, 3], abs=1e-9)
    assert k <= 3


# 14 ma trận đơn vị: s = r, alpha = 1 -> đúng 1 bước
def test_cg_ma_tran_don_vi():
    b = [3.0, -1.0, 2.0]
    x, k, _ = bat_output(np.eye(3), b, [0, 0, 0], 1e-10)
    assert x == pytest.approx(b)
    assert k == 1


# 15 ma trận 1x1
def test_cg_1x1():
    # 2x = 4 -> x = 2, r0 = 4, s0 = 4, alpha = 16/32 = 0.5, x1 = 0 + 0.5*4 = 2
    x, k, _ = bat_output([[2.0]], [4.0], [0.0], 1e-10)
    assert x == pytest.approx([2.0])
    assert k == 1


# 16 nghiệm ban đầu đã là nghiệm -> 0 lần lặp
def test_cg_x0_da_la_nghiem():
    x0 = [1 / 11, 7 / 11]
    x, k, _ = bat_output([[4, 1], [1, 3]], [1, 2], x0, 1e-6)
    assert k == 0
    assert x == pytest.approx(x0)


# 17 x0 khác 0 vẫn hội tụ về cùng nghiệm
def test_cg_x0_khac_khong():
    x, k, _ = bat_output([[4, 1], [1, 3]], [1, 2], [5.0, -3.0], 1e-8)
    assert x == pytest.approx([1 / 11, 7 / 11], abs=1e-8)
    assert k <= 2


# 18 so sánh với np.linalg.solve trên 300 ma trận SPD ngẫu nhiên
def test_cg_so_voi_numpy_ngau_nhien():
    rng = np.random.default_rng(0)
    for _ in range(300):
        n = int(rng.integers(2, 7))
        B = rng.standard_normal((n, n))
        A = B @ B.T + n * np.eye(n)     # B B^T đối xứng nửa xác định dương, cộng n*I để xác định dương
        b = rng.standard_normal(n)
        x, k, _ = bat_output(A, b, np.zeros(n), 1e-8)
        assert x == pytest.approx(np.linalg.solve(A, b), abs=1e-6)
        assert 1 <= k <= n
        assert np.linalg.norm(b - A @ x) < 1e-6   # phần dư đủ nhỏ


# 19 eps lớn -> dừng ngay từ đầu
def test_cg_eps_lon():
    # ||r0|| = căn5 = 2.236 < 100
    x, k, _ = bat_output([[4, 1], [1, 3]], [1, 2], [0, 0], 100.0)
    assert k == 0


# 20 không làm thay đổi x0 đầu vào (hàm dùng .copy())
def test_cg_khong_doi_x0():
    x0 = np.array([1.0, 1.0])
    goc = x0.copy()
    bat_output([[4, 1], [1, 3]], [1, 2], x0, 1e-8)
    assert np.array_equal(x0, goc)


# 21 đầu vào là list Python (hàm dùng np.asarray)
def test_cg_dau_vao_list():
    x, k, _ = bat_output([[2, 0], [0, 2]], [2, 4], [0, 0], 1e-10)
    assert x == pytest.approx([1.0, 2.0])


# 22 kiểu dữ liệu trả về
def test_cg_kieu_tra_ve():
    x, k, _ = bat_output([[4, 1], [1, 3]], [1, 2], [0, 0], 1e-8)
    assert isinstance(x, np.ndarray)
    assert isinstance(k, int)


# 23 nội dung in ra màn hình (xem thêm PHẦN 3)
def test_cg_in_ra_man_hinh():
    _, _, out = bat_output([[4, 1], [1, 3]], [1, 2], [0, 0], 1e-8)
    assert "Nghiệm ban đầu x0" in out
    assert "Lần lặp thứ 1" in out
    assert "Thỏa mãn sau" in out


def test_cg_in_ra_khi_x0_da_dat():
    _, _, out = bat_output([[4, 1], [1, 3]], [1, 2], [1 / 11, 7 / 11], 1e-6)
    assert "Nghiệm ban đầu đã thỏa mãn" in out


# 24-30 Các trường hợp phải báo lỗi
def test_cg_khong_vuong():
    with pytest.raises(ValueError, match="vuông"):
        bat_output([[1, 2, 3], [4, 5, 6]], [1, 2], [0, 0], 1e-6)


def test_cg_ma_tran_rong():
    with pytest.raises(ValueError, match="vuông"):
        bat_output(np.empty((0, 0)), [], [], 1e-6)


def test_cg_khong_phai_ma_tran_2_chieu():
    with pytest.raises(ValueError, match="vuông"):
        bat_output([1, 2, 3], [1, 2, 3], [0, 0, 0], 1e-6)


def test_cg_khong_doi_xung():
    # A != A^T
    with pytest.raises(ValueError, match="đối xứng"):
        bat_output([[2, 1], [0, 2]], [1, 1], [0, 0], 1e-6)


def test_cg_doi_xung_nhung_khong_xac_dinh_duong():
    # A = 1  2 -> định thức con cấp 2 = 1 - 4 = -3 < 0
    #     2  1
    with pytest.raises(ValueError, match="xác định dương"):
        bat_output([[1, 2], [2, 1]], [1, 1], [0, 0], 1e-6)


def test_cg_xac_dinh_am():
    # định thức con cấp 1 = -1 < 0
    with pytest.raises(ValueError, match="xác định dương"):
        bat_output([[-1, 0], [0, -1]], [1, 1], [0, 0], 1e-6)


def test_cg_nua_xac_dinh_duong_suy_bien():
    # A = 1 1 -> định thức con cấp 2 = 1 - 1 = 0 -> không xác định dương
    #     1 1
    with pytest.raises(ValueError, match="xác định dương"):
        bat_output([[1, 1], [1, 1]], [1, 1], [0, 0], 1e-6)

# PHẦN 3: OUTPUT (nội dung in ra màn hình)

# Với A = [[4,1],[1,3]], b = [1,2], x0 = [0,0], eps = 1e-8 chương trình in ra:
#
#   Nghiệm ban đầu x0: [0, 0]
#   Phần dư ban đầu r0: [1. 2.]
#
#   Lần lặp thứ 1:
#   Nghiệm x = [0.25, 0.5]
#   Phần dư r = [-0.5, 0.25]
#   Chuẩn phần dư ||r|| = 0.559017
#
#   Lần lặp thứ 2:
#   Nghiệm x = [0.090909, 0.636364]
#   Phần dư r = [0.0, 0.0]
#   Chuẩn phần dư ||r|| = 0.0
#   Thỏa mãn sau 2 lần lặp.
#
# (giá trị lần 1 đã tính tay ở test 12; lần 2: x = (1/11, 7/11) = (0.090909, 0.636364))

A_MAU = [[4, 1], [1, 3]]
B_MAU = [1, 2]


# 31 output đầy đủ của ví dụ chuẩn (so khớp từng dòng)
def test_output_vi_du_chuan():
    _, _, out = bat_output(A_MAU, B_MAU, [0, 0], 1e-8)
    dong = lay_dong(out)
    assert dong[0] == "Nghiệm ban đầu x0: [0, 0]"
    assert dong[1].startswith("Phần dư ban đầu r0:")
    assert dong[2:] == [
        "Lần lặp thứ 1:",
        "Nghiệm x = [0.25, 0.5]",
        "Phần dư r = [-0.5, 0.25]",
        "Chuẩn phần dư ||r|| = 0.559017",
        "Lần lặp thứ 2:",
        "Nghiệm x = [0.090909, 0.636364]",
        "Phần dư r = [0.0, 0.0]",
        "Chuẩn phần dư ||r|| = 0.0",
        "Thỏa mãn sau 2 lần lặp.",
    ]


# 32 các lần lặp được in đúng thứ tự, không có lần lặp thừa
def test_output_thu_tu_lan_lap():
    _, _, out = bat_output(A_MAU, B_MAU, [0, 0], 1e-8)
    assert out.index("Lần lặp thứ 1") < out.index("Lần lặp thứ 2")
    assert "Lần lặp thứ 3" not in out   # n = 2 nên tối đa 2 lần lặp


# 33 dừng sớm: chỉ in 1 lần lặp rồi báo "sau 1 lần lặp"
def test_output_dung_som():
    # eps = 0.6 > 0.559017 = ||r1|| (xem test 12)
    _, _, out = bat_output(A_MAU, B_MAU, [0, 0], 0.6)
    assert "Chuẩn phần dư ||r|| = 0.559017" in out
    assert "Thỏa mãn sau 1 lần lặp." in out
    assert "Lần lặp thứ 2" not in out


# 34 x0 đã là nghiệm: không in lần lặp nào, chỉ báo thỏa mãn
def test_output_x0_da_dat():
    _, _, out = bat_output(A_MAU, B_MAU, [1 / 11, 7 / 11], 1e-6)
    assert "Nghiệm ban đầu đã thỏa mãn điều kiện dừng." in out
    assert "Lần lặp thứ" not in out
    assert "Thỏa mãn sau" not in out


# 35 ma trận đơn vị: A = I, b = (3,-1) -> x = b ngay sau 1 lần lặp
def test_output_ma_tran_don_vi():
    _, _, out = bat_output([[1, 0], [0, 1]], [3, -1], [0, 0], 1e-10)
    dong = lay_dong(out)
    assert "Nghiệm x = [3.0, -1.0]" in dong
    assert "Chuẩn phần dư ||r|| = 0.0" in dong
    assert dong[-1] == "Thỏa mãn sau 1 lần lặp."


# 36 số lần "Lần lặp thứ" in ra = k trả về; nghiệm in ở lần cuối = nghiệm trả về (làm tròn 6 số)
def test_output_khop_gia_tri_tra_ve():
    rng = np.random.default_rng(1)
    for _ in range(100):
        n = int(rng.integers(2, 6))
        B = rng.standard_normal((n, n))
        A = B @ B.T + n * np.eye(n)
        b = rng.standard_normal(n)
        x, k, out = bat_output(A, b, np.zeros(n), 1e-3)
        dong = lay_dong(out)

        dong_lap = [d for d in dong if d.startswith("Lần lặp thứ")]
        assert len(dong_lap) == k

        dong_x = [d for d in dong if d.startswith("Nghiệm x =")]
        in_ra = eval(dong_x[-1].split("=", 1)[1])      # chuỗi "[...]" -> list số
        assert in_ra == pytest.approx([round(float(v), 6) for v in x])

        dong_chuan = [d for d in dong if d.startswith("Chuẩn phần dư")]
        chuan_cuoi = float(dong_chuan[-1].split("=")[1])
        assert chuan_cuoi < 1e-3                       # lần lặp cuối thỏa điều kiện dừng


# 37 đầu vào sai -> báo lỗi trước khi in bất kỳ lần lặp nào
def test_output_khong_in_khi_bao_loi():
    buf = io.StringIO()
    with pytest.raises(ValueError):
        with contextlib.redirect_stdout(buf):
            conjugate_gradient([[1, 2], [2, 1]], [1, 1], [0, 0], 1e-6)  # không xác định dương
    out = buf.getvalue()
    assert "Lần lặp thứ" not in out
    assert "Nghiệm ban đầu x0" not in out


# CHẠY TẤT CẢ TEST

test_functions = [
    test_dinh_thuc_2x2, test_dinh_thuc_1x1, test_dinh_thuc_duong_cheo,
    test_dinh_thuc_don_vi, test_dinh_thuc_doi_dong, test_dinh_thuc_suy_bien,
    test_dinh_thuc_so_voi_numpy, test_dinh_thuc_khong_doi_ma_tran_goc, test_dinh_thuc_tol,
    test_dinh_thuc_bao_loi, test_cg_vi_du_chuan, test_cg_mot_buoc_lap_tinh_tay,
    test_cg_duong_cheo, test_cg_ma_tran_don_vi, test_cg_1x1,
    test_cg_x0_da_la_nghiem, test_cg_x0_khac_khong, test_cg_so_voi_numpy_ngau_nhien,
    test_cg_eps_lon, test_cg_khong_doi_x0, test_cg_dau_vao_list,
    test_cg_kieu_tra_ve, test_cg_in_ra_man_hinh, test_cg_in_ra_khi_x0_da_dat,
    test_cg_khong_vuong, test_cg_ma_tran_rong, test_cg_khong_phai_ma_tran_2_chieu,
    test_cg_khong_doi_xung, test_cg_doi_xung_nhung_khong_xac_dinh_duong, test_cg_xac_dinh_am,
    test_cg_nua_xac_dinh_duong_suy_bien, test_output_vi_du_chuan, test_output_thu_tu_lan_lap,
    test_output_dung_som, test_output_x0_da_dat, test_output_ma_tran_don_vi,
    test_output_khop_gia_tri_tra_ve, test_output_khong_in_khi_bao_loi,
]

so_pass = 0
for func in test_functions:
    try:
        with contextlib.redirect_stdout(io.StringIO()):   # ẩn output do chương trình CG tự in
            func()
        print(f"✔ {func.__name__} ---> PASSED")
        so_pass += 1
    except Exception as e:
        print(f"✘ {func.__name__} ---> FAILED: {type(e).__name__}: {e}")

print(f"\n{so_pass}/{len(test_functions)} test PASSED")
if so_pass == len(test_functions):
    print("pass")
assert so_pass == len(test_functions), "Có ca kiểm thử gradient liên hợp thất bại."


✔ test_dinh_thuc_2x2 ---> PASSED
✔ test_dinh_thuc_1x1 ---> PASSED
✔ test_dinh_thuc_duong_cheo ---> PASSED
✔ test_dinh_thuc_don_vi ---> PASSED
✔ test_dinh_thuc_doi_dong ---> PASSED
✔ test_dinh_thuc_suy_bien ---> PASSED
✔ test_dinh_thuc_so_voi_numpy ---> PASSED
✔ test_dinh_thuc_khong_doi_ma_tran_goc ---> PASSED
✔ test_dinh_thuc_tol ---> PASSED
✔ test_dinh_thuc_bao_loi ---> PASSED
✔ test_cg_vi_du_chuan ---> PASSED
✔ test_cg_mot_buoc_lap_tinh_tay ---> PASSED
✔ test_cg_duong_cheo ---> PASSED
✔ test_cg_ma_tran_don_vi ---> PASSED
✔ test_cg_1x1 ---> PASSED
✔ test_cg_x0_da_la_nghiem ---> PASSED
✔ test_cg_x0_khac_khong ---> PASSED
✔ test_cg_so_voi_numpy_ngau_nhien ---> PASSED
✔ test_cg_eps_lon ---> PASSED
✔ test_cg_khong_doi_x0 ---> PASSED
✔ test_cg_dau_vao_list ---> PASSED
✔ test_cg_kieu_tra_ve ---> PASSED
✔ test_cg_in_ra_man_hinh ---> PASSED
✔ test_cg_in_ra_khi_x0_da_dat ---> PASSED
✔ test_cg_khong_vuong ---> PASSED
✔ test_cg_ma_tran_rong ---> PASSED
✔ test_cg_khong_phai_ma_tran_2_chieu ---> P